In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df=pd.read_csv('qoute_dataset.csv')

In [ ]:
df.shape

(3038, 2)

In [ ]:
df.head()

,quote,Author
0,“The world as we have created it is a process ...,Albert Einstein
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling
2,“There are only two ways to live your life. On...,Albert Einstein
3,"“The person, be it gentleman or lady, who has ...",Jane Austen
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe


In [ ]:
quote=df['quote']

In [ ]:
quote.head()

,quote
0,“The world as we have created it is a process ...
1,"“It is our choices, Harry, that show what we t..."
2,“There are only two ways to live your life. On...
3,"“The person, be it gentleman or lady, who has ..."
4,"“Imperfection is beauty, madness is genius and..."


In [ ]:
print(type(quote))

<class 'pandas.core.series.Series'>


In [ ]:
quote=quote.str.lower()

In [ ]:
quote[0]

'“the world as we have created it is a process of our thinking. it cannot be changed without changing our thinking.”'

In [ ]:
import string
punc=string.punctuation + '“”‘’'

translator=str.maketrans('','',punc)


In [ ]:
quote=quote.apply(lambda x: x.translate(translator))

In [ ]:
quote.head()

,quote
0,the world as we have created it is a process o...
1,it is our choices harry that show what we trul...
2,there are only two ways to live your life one ...
3,the person be it gentleman or lady who has not...
4,imperfection is beauty madness is genius and i...


In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer

In [ ]:
vocab_size=8770

tok=Tokenizer(num_words=vocab_size, oov_token='<OOV>')
tok.fit_on_texts(quote)

In [ ]:
word_index=tok.word_index

In [ ]:
len(word_index)

8770

In [ ]:
list(word_index.items())[:10]

[('<OOV>', 1),
 ('the', 2),
 ('you', 3),
 ('to', 4),
 ('and', 5),
 ('a', 6),
 ('i', 7),
 ('is', 8),
 ('of', 9),
 ('that', 10)]

In [ ]:
sequence=tok.texts_to_sequences(quote)

In [ ]:
sequence[0]

[2,
 63,
 30,
 20,
 17,
 940,
 11,
 8,
 6,
 1144,
 9,
 71,
 286,
 11,
 147,
 13,
 806,
 106,
 744,
 71,
 286]

In [ ]:
X=[]
y=[]

for seq in sequence:
  for i in range(1,len(seq)):
    X.append(seq[:i])
    y.append(seq[i])

In [ ]:
X[:5]

[[2], [2, 63], [2, 63, 30], [2, 63, 30, 20], [2, 63, 30, 20, 17]]

In [ ]:
y[:5]

[63, 30, 20, 17, 940]

In [ ]:
len(X)

85269

In [ ]:
len(y)

85269

In [ ]:
max_len=max(len(x) for x in X)

In [ ]:
max_len

745

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [ ]:
X_padded=pad_sequences(X,maxlen=max_len, padding='pre')

In [ ]:
X_padded[:5]

array([[ 0,  0,  0, ...,  0,  0,  2],
       [ 0,  0,  0, ...,  0,  2, 63],
       [ 0,  0,  0, ...,  2, 63, 30],
       [ 0,  0,  0, ..., 63, 30, 20],
       [ 0,  0,  0, ..., 30, 20, 17]], dtype=int32)

In [ ]:
len(X_padded[0])

745

In [ ]:
type(y)

list

In [ ]:
y=np.array(y)

In [ ]:
X_padded.shape

(85269, 745)

In [ ]:
y.shape

(85269,)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Dense, Dropout, SimpleRNN, LSTM

In [ ]:
embedding_dim=50
rnn_units=128

In [ ]:
rnn_model=Sequential()
rnn_model.add(
    Embedding(input_dim=vocab_size,output_dim=embedding_dim, input_length=max_len)
)
rnn_model.add(SimpleRNN(units=rnn_units,))
rnn_model.add(Dense(units=vocab_size, activation='softmax'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [ ]:
rnn_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [ ]:
rnn_model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_1 (SimpleRNN)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
lstm_model=Sequential()
lstm_model.add(
    Embedding(input_dim=vocab_size,output_dim=embedding_dim, input_length=max_len)
)
lstm_model.add(
    LSTM(units=rnn_units)
)
lstm_model.add(
    Dense(units=vocab_size,activation='softmax')
)

In [ ]:
lstm_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [ ]:
lstm_model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
epochs=10
batch_size=64

In [ ]:
#rnn_model.fit(X_padded,y,epochs=epochs, batch_size=batch_size, validation_split=0.1)

In [ ]:
lstm_model.save('rnn_model.h5')

In [ ]:
epochs=100
batch_size=64

In [ ]:
lstm_model.fit(X_padded,y,epochs=epochs, batch_size=batch_size, validation_split=0.1)

Epoch 1/100
1200/1200 ━━━━━━━━━━━━━━━━━━━━ 50s 37ms/step - accuracy: 0.0437 - loss: 6.6694 - val_accuracy: 0.0566 - val_loss: 6.5896
Epoch 2/100
1200/1200 ━━━━━━━━━━━━━━━━━━━━ 44s 37ms/step - accuracy: 0.0726 - loss: 6.1435 - val_accuracy: 0.0863 - val_loss: 6.3746
Epoch 3/100
1200/1200 ━━━━━━━━━━━━━━━━━━━━ 44s 36ms/step - accuracy: 0.0978 - loss: 5.8123 - val_accuracy: 0.0998 - val_loss: 6.3278
Epoch 4/100
1200/1200 ━━━━━━━━━━━━━━━━━━━━ 43s 36ms/step - accuracy: 0.1142 - loss: 5.5401 - val_accuracy: 0.1047 - val_loss: 6.3100
Epoch 5/100
1200/1200 ━━━━━━━━━━━━━━━━━━━━ 43s 36ms/step - accuracy: 0.1297 - loss: 5.2908 - val_accuracy: 0.1123 - val_loss: 6.3580
Epoch 6/100
1200/1200 ━━━━━━━━━━━━━━━━━━━━ 43s 36ms/step - accuracy: 0.1422 - loss: 5.0624 - val_accuracy: 0.1143 - val_loss: 6.4519
Epoch 7/100
1200/1200 ━━━━━━━━━━━━━━━━━━━━ 82s 36ms/step - accuracy: 0.1529 - loss: 4.8478 - val_accuracy: 0.1159 - val_loss: 6.4965
Epoch 8/100
1200/1200 ━━━━━━━━━━━━━━━━━━━━ 43s 36ms/step - accuracy: 

In [ ]:
lstm_model.save('LSTM_model.h5')



In [ ]:
#from tensorflow.keras.models import load_model

#lstm_model = load_model("LSTM_model.h5")

In [ ]:
index_to_word={}
for word, index in word_index.items():
  index_to_word[index]=word

In [ ]:
def predictor(model,tokenizer,text,max_len):
  text=text.lower()
  text=text.translate(translator)
  seq=tokenizer.texts_to_sequences([text])[0]
  seq=pad_sequences([seq],maxlen=max_len,padding='pre')
  pred=model.predict(seq,verbose=0)
  pred_word_index=np.argmax(pred)
  pred_word=index_to_word[pred_word_index]
  return pred_word

In [ ]:
text='school is'
next_word=predictor(lstm_model,tok,text,max_len)
print(next_word)

great


In [ ]:
def generate_text(model,tokenizer,text,max_len,n_words):
  for _ in range(n_words):
    nex_word=predictor(model,tokenizer,text,max_len)
    if next_word == "":
      break
    text+=' '+nex_word
  return text

In [ ]:
txt="The meaning of life"

res=generate_text(lstm_model,tok,txt,max_len,10)
print(res)

The meaning of life is music with the risk it the joy of a


In [ ]:
txt="are you a"

res=generate_text(lstm_model,tok,txt,max_len,10)
print(res)

are you a human being a way a dream to power power the


In [ ]:
import pickle

with open("tokenizer.pkl","wb") as f:
  pickle.dump(tok,f)

In [ ]:
with open("max_len.pkl","wb") as f:
  pickle.dump(max_len,f)